# VASP 参数收敛结果分析

**适用环境：Windows / macOS / Linux（Python 分析）；Linux + PBS（计算）。**

完整操作与实际结果图见[VASP 参数收敛测试](VASP参数收敛测试.md)。本 Notebook 读取案例脚本产生的 JSON，不生成或提交计算任务。


## 读取结果

先完成[最小案例](../../../examples/vasp/minimal/parameter-convergence/)的 PBS 作业，将 `results_path` 指向作业目录中的 `convergence-results.json`。默认路径适用于直接在案例目录完成计算的情况。


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np

repo = next(parent for parent in [Path.cwd(), *Path.cwd().parents]
            if (parent / "examples/vasp/minimal").is_dir())
results_path = repo / "examples/vasp/minimal/parameter-convergence/convergence-results.json"


In [ ]:
result = json.loads(results_path.read_text())
for key in ["cutoff_points", "k_points"]:
    print(key)
    for point in result[key]:
        print(point)
print("截断能最后两点差 (eV/atom):", result["cutoff_last_change_eV_atom"])
print("k 网格最后两点差 (eV/atom):", result["k_mesh_last_change_eV_atom"])


## 比较趋势

每组以扫描的最后一个点为能量参考。比较时保持另一参数、结构和电子精度一致；最后两点接近，不等于所有性质都已收敛。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5), constrained_layout=True)
for ax, key, xkey, xlabel in [
    (axes[0], "cutoff_points", "cutoff_eV", "Cutoff (eV)"),
    (axes[1], "k_points", "k_mesh", "k mesh per direction"),
]:
    points = result[key]
    energy = np.array([p["energy_eV_atom"] for p in points])
    ax.plot([p[xkey] for p in points], (energy - energy[-1]) * 1000, "o-")
    ax.set(xlabel=xlabel, ylabel="Energy relative to last point (meV/atom)")
    ax.grid(alpha=0.2)
plt.show()


## 选择参数

结合连续多个点的变化和目标量选择候选参数，再做联合复核。力、应力、能隙和输运等性质需有自己的判据；不要把硅的小案例取值直接当作 GeTe 或其他材料的推荐值。
